# Дообучение эмбеддера v2 — исправленная версия

### Что исправлено по сравнению с v1

1. **Ложные негативы (главная причина деградации).** В датасете ~268 уникальных документов на 3000 запросов (≈11 запросов на документ). `MultipleNegativesRankingLoss` использует in-batch negatives: все позитивы других пар в батче = негативы. Если 2 запроса в батче указывают на один позитив → модель учится ОТТАЛКИВАТЬ правильный документ. **Решение:** дедупликация — оставляем 1 лучший запрос на уникальный позитив, плюс чистый hard negative. Это даёт ~268 чистых троек без ложных негативов.

2. **Слишком высокий LR → catastrophic forgetting.** 2e-5 на маленьком датасете с 3 эпохами — модель забывает общие знания. **Решение:** LR = 5e-6, 2 эпохи, увеличенный warmup.

3. **Eval-корпус слишком мал.** Evaluator строил корпус только из eval-split (~30 документов). **Решение:** в корпус eval попадают ВСЕ уникальные документы из всего датасета, а eval-запросы — hold-out подмножество.

4. **Добавлена стратегия 2:** если после дедупликации хочется использовать ВСЕ 3000 строк — переключаемся на `TripletLoss`, который не создаёт in-batch negatives и безопасен при дублирующихся позитивах.

## 0. Окружение

In [1]:
!nvidia-smi

Sun May 24 13:30:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U "sentence-transformers>=3.3.0" "transformers>=4.51.0" "datasets>=2.19.0" "accelerate>=0.30.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.2 MB/s eta 0:00:00


In [3]:
import os
import json
import random
import hashlib
import logging
from pathlib import Path
from collections import defaultdict

import torch
import numpy as np
import pandas as pd
from datasets import Dataset

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    evaluation,
)
from sentence_transformers.training_args import BatchSamplers

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM, GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

Device: cuda
GPU: Tesla T4
VRAM, GB: 15.6


/tmp/ipykernel_1215/908020164.py:14: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
/tmp/ipykernel_1215/908020164.py:14: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import (
/tmp/ipykernel_1215/908020164.py:21: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


## 1. Модель и префиксы

In [4]:
BASE_MODEL = "intfloat/multilingual-e5-base"
OUTPUT_DIR = f"./finetuned-{BASE_MODEL.split('/')[-1]}-legal-ru-v2"

def get_prefix_fn(model_name: str):
    name = model_name.lower()
    if "e5" in name:
        return (lambda q: f"query: {q}", lambda d: f"passage: {d}")
    if "bge" in name and "m3" in name:
        return (lambda q: q, lambda d: d)
    if "qwen3-embedding" in name:
        instr = "Given a legal question in Russian, retrieve relevant passages from traffic regulations and laws"
        return (lambda q: f"Instruct: {instr}\nQuery: {q}", lambda d: d)
    if "rosberta" in name:
        return (lambda q: f"search_query: {q}", lambda d: f"search_document: {d}")
    return (lambda q: q, lambda d: d)

format_query, format_doc = get_prefix_fn(BASE_MODEL)
print("Query prefix:", format_query("тест"))
print("Doc prefix:",   format_doc("тест"))

Query prefix: query: тест
Doc prefix: passage: тест


## 2. Загрузка и дедупликация датасета

### Почему это критично

`MultipleNegativesRankingLoss` (MNRL) работает так: в батче из N пар `(q_i, p_i)` для каждого `q_i` все `p_j (j ≠ i)` считаются негативами. Если два запроса `q_1` и `q_2` в одном батче указывают на один и тот же документ `p`, то:
- для `q_1` документ `p` помечен как позитив, но одновременно (как позитив `q_2`) он попадает в негативы
- модель получает **противоречивый сигнал** и деградирует

**Решение:** оставить только 1 запрос на каждый уникальный документ. Выбираем самый длинный и информативный запрос.

In [5]:
DATA_PATH = "/content/train-2.jsonl"

# ========== Выбор стратегии ==========
# "mnrl_dedup"  — дедуплицируем позитивы (1 запрос на документ), используем MNRL. Меньше данных, но чистый сигнал.
# "triplet_full" — используем ВСЕ 3000 строк, но с TripletLoss (не создаёт in-batch negatives). Больше данных.
STRATEGY = "mnrl_dedup"   # <--- переключай тут

print(f"Стратегия: {STRATEGY}")

Стратегия: mnrl_dedup


In [6]:
# Загрузка через pandas (обход проблемы с кодировкой в load_dataset)
df = pd.read_json(DATA_PATH, lines=True, encoding="utf-8")
print(f"Загружено строк: {len(df)}")
print(f"Колонки: {list(df.columns)}")

# Считаем статистику дублирования
pos_counts = df["positive"].value_counts()
print(f"\nУникальных позитивов: {len(pos_counts)}")
print(f"Макс. запросов на 1 позитив: {pos_counts.max()}")
print(f"Среднее: {pos_counts.mean():.1f}")
print(f"\nТоп-5 дублированных позитивов (сколько запросов указывают на них):")
for pos_text, cnt in pos_counts.head(5).items():
    print(f"  {cnt}x : {pos_text[:120]}...")

Загружено строк: 3000
Колонки: ['query', 'positive', 'negative']

Уникальных позитивов: 743
Макс. запросов на 1 позитив: 12
Среднее: 4.0

Топ-5 дублированных позитивов (сколько запросов указывают на них):
  12x : Выезд на перекресток или пересечение проезжей части дороги в случае образовавшегося затора, который вынудил водителя ост...
  11x : Проезд на запрещающий сигнал светофора или на запрещающий жест регулировщика, за исключением случаев, предусмотренных ча...
  10x : За административные правонарушения, предусмотренные настоящей статьей, лица, осуществляющие предпринимательскую деятельн...
  10x : Нарушение правил проезда через железнодорожные переезды, за исключением случаев, предусмотренных частью 1 настоящей стат...
  10x : Движение на грузовом автомобиле с разрешенной максимальной массой более 3,5 тонны по автомагистрали далее второй полосы,...


In [7]:
if STRATEGY == "mnrl_dedup":
    # ============ ДЕДУПЛИКАЦИЯ ============
    # Для каждого уникального позитива оставляем 1 запрос.
    # Критерий: берём самый длинный запрос (обычно самый информативный)
    # и самый длинный негатив (самый hard).

    best_per_positive = {}
    for _, row in df.iterrows():
        pos_key = row["positive"]
        if pos_key not in best_per_positive:
            best_per_positive[pos_key] = row
        else:
            # Берём запрос, который длиннее и информативнее
            existing = best_per_positive[pos_key]
            if len(row["query"]) > len(existing["query"]):
                best_per_positive[pos_key] = row

    deduped = pd.DataFrame(list(best_per_positive.values()))
    print(f"После дедупликации: {len(deduped)} строк (было {len(df)})")
    print(f"Уникальных позитивов: {deduped['positive'].nunique()}")

    work_df = deduped
else:
    # ============ ВСЕ ДАННЫЕ (для TripletLoss) ============
    work_df = df.copy()
    print(f"Используем все {len(work_df)} строк с TripletLoss")

print(f"\nРазмер рабочего датасета: {len(work_df)}")

После дедупликации: 743 строк (было 3000)
Уникальных позитивов: 743

Размер рабочего датасета: 743


In [8]:
# Применяем префиксы
def apply_prefixes(row):
    return {
        "anchor":   format_query(row["query"]),
        "positive": format_doc(row["positive"]),
        "negative": format_doc(row["negative"]),
    }

work_prefixed = work_df.apply(apply_prefixes, axis=1, result_type="expand")

# Train/eval split — разделяем по ПОЗИТИВАМ, а не по строкам,
# чтобы eval содержал документы, которых модель не видела при обучении.
unique_positives = work_prefixed["positive"].unique()
np.random.shuffle(unique_positives)
n_eval = max(20, int(len(unique_positives) * 0.15))  # 15% позитивов в eval
eval_positives = set(unique_positives[:n_eval])
train_positives = set(unique_positives[n_eval:])

train_mask = work_prefixed["positive"].isin(train_positives)
eval_mask  = work_prefixed["positive"].isin(eval_positives)

train_df = work_prefixed[train_mask].reset_index(drop=True)
eval_df  = work_prefixed[eval_mask].reset_index(drop=True)

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
eval_ds  = Dataset.from_pandas(eval_df, preserve_index=False)

print(f"Train: {len(train_ds)} строк ({len(train_positives)} уник. позитивов)")
print(f"Eval:  {len(eval_ds)} строк ({len(eval_positives)} уник. позитивов)")
print(f"Пересечение позитивов train∩eval: {len(train_positives & eval_positives)} (должно быть 0)")

Train: 632 строк (632 уник. позитивов)
Eval:  111 строк (111 уник. позитивов)
Пересечение позитивов train∩eval: 0 (должно быть 0)


## 3. Eval: полный корпус

Корпус для `InformationRetrievalEvaluator` строим из **всех** уникальных документов (train + eval), а eval-запросы — только из eval-split. Это имитирует реальную RAG-систему: модель ищет по всему корпусу, а не по горстке eval-документов.

In [9]:
def build_full_corpus_evaluator(eval_data, all_data, name="legal-eval"):
    """
    Корпус = ВСЕ уникальные позитивы + негативы из всего датасета.
    Запросы = только из eval-split.
    """
    # 1) Собираем полный корпус
    corpus = {}  # doc_id -> text
    doc_text_to_id = {}  # text -> doc_id (дедупликация)

    def add_to_corpus(text):
        if text not in doc_text_to_id:
            doc_id = f"d{len(doc_text_to_id)}"
            doc_text_to_id[text] = doc_id
            corpus[doc_id] = text
        return doc_text_to_id[text]

    # Добавляем все документы из всего датасета
    for _, row in all_data.iterrows():
        add_to_corpus(row["positive"])
        add_to_corpus(row["negative"])

    # 2) Запросы и релевантности — только из eval
    queries = {}
    relevant_docs = {}

    for i, row in eval_data.iterrows():
        qid = f"q{i}"
        queries[qid] = row["anchor"]
        pos_id = doc_text_to_id[row["positive"]]
        relevant_docs[qid] = {pos_id}

    print(f"Eval: {len(queries)} запросов, корпус: {len(corpus)} документов")

    return evaluation.InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        accuracy_at_k=[1, 3, 5, 10],
        precision_recall_at_k=[1, 3, 5, 10],
        mrr_at_k=[10],
        ndcg_at_k=[10],
        map_at_k=[10],
        show_progress_bar=True,
    )

ir_evaluator = build_full_corpus_evaluator(eval_df, work_prefixed)

Eval: 111 запросов, корпус: 853 документов


## 4. Бейзлайн

In [10]:
model = SentenceTransformer(BASE_MODEL, device=DEVICE)
model.max_seq_length = min(model.max_seq_length, 512)

print("Макс. длина:", model.max_seq_length)
print("Размерность эмбеддинга:", model.get_sentence_embedding_dimension())

baseline_metrics = ir_evaluator(model)
print("\n=== БЕЙЗЛАЙН ===")
for k in sorted(baseline_metrics.keys()):
    if any(m in k for m in ["ndcg@10", "mrr@10", "accuracy@1", "accuracy@5", "recall@10"]):
        print(f"  {k:55s} {baseline_metrics[k]:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:04<00:00,  4.69s/it]


=== БЕЙЗЛАЙН ===
  legal-eval_cosine_accuracy@1                            0.3063
  legal-eval_cosine_accuracy@10                           0.7027
  legal-eval_cosine_accuracy@5                            0.6577
  legal-eval_cosine_mrr@10                                0.4449
  legal-eval_cosine_ndcg@10                               0.5083
  legal-eval_cosine_recall@10                             0.7027


## 5. Лосс

Два варианта в зависимости от стратегии:

- **`mnrl_dedup`** — `MultipleNegativesRankingLoss`. Безопасен, потому что каждый позитив встречается только 1 раз → нет ложных негативов в батче.

- **`triplet_full`** — `TripletLoss`. Не использует in-batch negatives вообще — каждая тройка (anchor, positive, negative) обрабатывается независимо. Можно спокойно иметь дубли позитивов.

In [11]:
if STRATEGY == "mnrl_dedup":
    loss = losses.MultipleNegativesRankingLoss(model=model)
    print("Loss: MultipleNegativesRankingLoss (дедуплицированный датасет)")
elif STRATEGY == "triplet_full":
    loss = losses.TripletLoss(model=model, distance_metric=losses.TripletDistanceMetric.COSINE)
    print("Loss: TripletLoss (полный датасет, без in-batch negatives)")

Loss: MultipleNegativesRankingLoss (дедуплицированный датасет)


## 6. Параметры обучения

Ключевые отличия от v1:
- **LR = 5e-6** (было 2e-5) — мягкая адаптация без catastrophic forgetting
- **2 эпохи** (было 3) — меньше риск переобучения
- **warmup_ratio = 0.2** (было 0.1) — дольше прогреваемся
- **gradient_accumulation_steps = 4** с batch_size=8 → эффективный батч 32
- **eval каждые 50 шагов** (было 100) — датасет маленький, шагов мало

In [12]:
# Вычисляем количество шагов для адаптации eval_steps
effective_batch = 8 * 4  # per_device * accumulation
steps_per_epoch = max(1, len(train_ds) // effective_batch)
total_steps = steps_per_epoch * 2  # 2 эпохи
eval_steps = max(10, steps_per_epoch // 3)  # ~3 раза за эпоху

print(f"Размер train: {len(train_ds)}")
print(f"Эффективный батч: {effective_batch}")
print(f"Шагов/эпоха: {steps_per_epoch}")
print(f"Всего шагов: {total_steps}")
print(f"Eval каждые {eval_steps} шагов")

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,        # эффективный батч = 32
    learning_rate=5e-6,                   # ← ключевое: мягкий LR
    warmup_ratio=0.2,
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=2,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_legal-eval_cosine_ndcg@10",
    greater_is_better=True,
    report_to="none",
    gradient_checkpointing=False,
    seed=SEED,
)

Размер train: 632
Эффективный батч: 32
Шагов/эпоха: 19
Всего шагов: 38
Eval каждые 10 шагов


## 7. Обучение

In [19]:
torch.cuda.empty_cache()

In [20]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    loss=loss,
    evaluator=ir_evaluator,
)

trainer.train()

Step,Training Loss,Validation Loss,Legal-eval Cosine Accuracy@1,Legal-eval Cosine Accuracy@3,Legal-eval Cosine Accuracy@5,Legal-eval Cosine Accuracy@10,Legal-eval Cosine Precision@1,Legal-eval Cosine Precision@3,Legal-eval Cosine Precision@5,Legal-eval Cosine Precision@10,Legal-eval Cosine Recall@1,Legal-eval Cosine Recall@3,Legal-eval Cosine Recall@5,Legal-eval Cosine Recall@10,Legal-eval Cosine Ndcg@10,Legal-eval Cosine Mrr@10,Legal-eval Cosine Map@10
10,1.169772,1.561325,0.396396,0.657658,0.747748,0.855856,0.396396,0.219219,0.149550,0.085586,0.396396,0.657658,0.747748,0.855856,0.621957,0.547615,0.547615
20,0.940702,1.097176,0.495495,0.765766,0.846847,0.909910,0.495495,0.255255,0.169369,0.090991,0.495495,0.765766,0.846847,0.909910,0.708328,0.643075,0.643075
30,0.698585,0.899153,0.513514,0.819820,0.873874,0.927928,0.513514,0.273273,0.174775,0.092793,0.513514,0.819820,0.873874,0.927928,0.736754,0.673595,0.673595
40,0.648110,0.834065,0.540541,0.828829,0.891892,0.927928,0.540541,0.276276,0.178378,0.092793,0.540541,0.828829,0.891892,0.927928,0.749951,0.690952,0.690952


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:01<00:00,  1.44s/it]


...

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:01<00:00,  1.58s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

TrainOutput(global_step=40, training_loss=0.8642920851707458, metrics={'train_runtime': 216.553, 'train_samples_per_second': 5.837, 'train_steps_per_second': 0.185, 'total_flos': 0.0, 'train_loss': 0.8642920851707458, 'epoch': 2.0})

## 8. Сравнение

In [21]:
tuned_metrics = ir_evaluator(model)

print("\n=== СРАВНЕНИЕ ===")
print(f"{'metric':55s} {'baseline':>10s} {'tuned':>10s} {'delta':>10s}")
print("-" * 90)
for k in sorted(baseline_metrics.keys()):
    b = baseline_metrics[k]
    t = tuned_metrics.get(k, float("nan"))
    delta = t - b
    marker = "✅" if delta > 0.001 else ("❌" if delta < -0.001 else "  ")
    print(f"{marker} {k:53s} {b:10.4f} {t:10.4f} {delta:+10.4f}")

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:01<00:00,  1.51s/it]


=== СРАВНЕНИЕ ===
metric                                                    baseline      tuned      delta
------------------------------------------------------------------------------------------
✅ legal-eval_cosine_accuracy@1                              0.3063     0.5405    +0.2342
✅ legal-eval_cosine_accuracy@10                             0.7027     0.9279    +0.2252
✅ legal-eval_cosine_accuracy@3                              0.5676     0.8288    +0.2613
✅ legal-eval_cosine_accuracy@5                              0.6577     0.8919    +0.2342
✅ legal-eval_cosine_map@10                                  0.4449     0.6910    +0.2461
✅ legal-eval_cosine_mrr@10                                  0.4449     0.6910    +0.2461
✅ legal-eval_cosine_ndcg@10                                 0.5083     0.7500    +0.2416
✅ legal-eval_cosine_precision@1                             0.3063     0.5405    +0.2342
✅ legal-eval_cosine_precision@10                            0.0703     0.0928    +0.0225


## 9. Сохранение

In [22]:
FINAL_DIR = f"{OUTPUT_DIR}/final"
model.save(FINAL_DIR)
print("Сохранено в:", FINAL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Сохранено в: ./finetuned-multilingual-e5-base-legal-ru-v2/final


## 10. Если дедупликация дала мало данных — попробуй стратегию 2

Переставь в ячейке 2:
```python
STRATEGY = "triplet_full"
```
и перезапусти ноутбук. `TripletLoss` не использует in-batch negatives → безопасен при дублирующихся позитивах.

**Ожидаемый эффект:**
- `mnrl_dedup` — чистый сигнал, маленький датасет (~250–300 троек) → модель аккуратно адаптируется
- `triplet_full` — все 3000 троек, каждая с явным hard negative → модель видит больше примеров, но обучение медленнее

Рекомендую попробовать оба и сравнить.

## 11. Чеклист: что делать если метрики всё ещё не растут

1. **Ещё ниже LR:** попробуй `1e-6`
2. **1 эпоха** вместо 2
3. **Проверь правильность префиксов:** для E5 обязательны `query:` / `passage:`, для BGE — нет. Ошибка в префиксах = модель не понимает роли текста.
4. **Попробуй другую базовую модель:** `BAAI/bge-m3` часто лучше файн-тюнится на русском
5. **Hard negative mining:** после первой эпохи прогони запросы через модель и собери top-k ложных кандидатов как негативы для второй итерации
